## Pré-processamento é provavelmente a parte mais importante de ciência dos dados

Ter dados representativos sem atributos faltantes é provavelmente o pote de ouro em ciência dos dados. É muito incomum que os dados do mundo real não apresentem anomalias seja da própria natureza ou sejam anomalias introduzidas no processo de medição e registro da observação (amostra).

Esse notebook é voltado para como tratar dados mais complexos e transformar todas as informações em números que façam sentido para que o modelo seja capaz de traçar a relação entre atributos e classes. A seguir é oferecida uma pequena parcela de um conjunto de dados da empresa Porto Seguro, no qual uma competição foi aberta e os competidores foram desafiados a criar um modelo para prever se uma apólice teria um sinistro registrado ou não, indicando o uso do serviço.

Algumas características sobre o nome das features:
- O nome dos atributos indica o grupo ao qual pertence (ind, reg, car);
- Os prefixos bin e cat indicam atributos binários e categóricos, respectivamente;
- Atributos sem os prefixos citados podem ser ordinais ou contínuos;
- Atributos com -1 indicam dado faltante (missing); e
- A coluna 'target' indica se houve sinistro para apólice ou não.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go

In [2]:
dfTreino = pd.read_csv('F:/Documentos/train.csv')
dfTeste = pd.read_csv('F:/Documentos/test.csv')

In [3]:
headNumber = 5
print(f'Dataset de treino - Primeiras {headNumber} linhas')
display(dfTreino.head(headNumber))

print(f'Dataset de teste - Primeiras {headNumber} linhas')
display(dfTeste.head(headNumber))

Dataset de treino - Primeiras 5 linhas


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


Dataset de teste - Primeiras 5 linhas


,id,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,0,0,1,8,1,0,0,1,0,0,...,1,1,1,12,0,1,1,0,0,1
1,1,4,2,5,1,0,0,0,0,1,...,2,0,3,10,0,0,1,1,0,1
2,2,5,1,3,0,0,0,0,0,1,...,4,0,2,4,0,0,0,0,0,0
3,3,0,1,6,0,0,1,0,0,0,...,5,1,0,5,1,0,1,0,0,0
4,4,5,1,7,0,0,0,0,0,1,...,4,0,0,4,0,1,1,0,0,1


In [4]:
print('Dataset de treino - Estatistica descritiva')
display(dfTreino.describe())

print('Dataset de teste - Estatistica descritiva')
display(dfTeste.describe())

Dataset de treino - Estatistica descritiva


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,5.952120e+05,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,...,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000
mean,7.438036e+05,0.036448,1.900378,1.358943,4.423318,0.416794,0.405188,0.393742,0.257033,0.163921,...,5.441382,1.441918,2.872288,7.539026,0.122427,0.627840,0.554182,0.287182,0.349024,0.153318
std,4.293678e+05,0.187401,1.983789,0.664594,2.699902,0.493311,1.350642,0.488579,0.436998,0.370205,...,2.332871,1.202963,1.694887,2.746652,0.327779,0.483381,0.497056,0.452447,0.476662,0.360295
min,7.000000e+00,0.000000,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.719915e+05,0.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.435475e+05,0.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.000000,1.000000,3.000000,7.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,1.115549e+06,0.000000,3.000000,2.000000,6.000000,1.000000,0.000000,1.000000,1.000000,0.000000,...,7.000000,2.000000,4.000000,9.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.488027e+06,1.000000,7.000000,4.000000,11.000000,1.000000,6.000000,1.000000,1.000000,1.000000,...,19.000000,10.000000,13.000000,23.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Dataset de teste - Estatistica descritiva


,id,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,8.928160e+05,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,...,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000
mean,7.441535e+05,1.902371,1.358613,4.413734,0.417361,0.408132,0.393246,0.257191,0.163659,0.185905,...,5.438478,1.440265,2.875013,7.540367,0.123720,0.627756,0.554660,0.287796,0.349344,0.152428
std,4.296830e+05,1.986503,0.663002,2.700149,0.493453,1.355068,0.488471,0.437086,0.369966,0.389030,...,2.330081,1.200620,1.694072,2.745882,0.329262,0.483403,0.497004,0.452736,0.476763,0.359435
min,0.000000e+00,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.720218e+05,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.443070e+05,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.000000,1.000000,3.000000,7.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,1.116308e+06,3.000000,2.000000,6.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,...,7.000000,2.000000,4.000000,9.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.488026e+06,7.000000,4.000000,11.000000,1.000000,6.000000,1.000000,1.000000,1.000000,1.000000,...,20.000000,11.000000,15.000000,28.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
dfTreino.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595212 entries, 0 to 595211
Data columns (total 59 columns):
id                595212 non-null int64
target            595212 non-null int64
ps_ind_01         595212 non-null int64
ps_ind_02_cat     595212 non-null int64
ps_ind_03         595212 non-null int64
ps_ind_04_cat     595212 non-null int64
ps_ind_05_cat     595212 non-null int64
ps_ind_06_bin     595212 non-null int64
ps_ind_07_bin     595212 non-null int64
ps_ind_08_bin     595212 non-null int64
ps_ind_09_bin     595212 non-null int64
ps_ind_10_bin     595212 non-null int64
ps_ind_11_bin     595212 non-null int64
ps_ind_12_bin     595212 non-null int64
ps_ind_13_bin     595212 non-null int64
ps_ind_14         595212 non-null int64
ps_ind_15         595212 non-null int64
ps_ind_16_bin     595212 non-null int64
ps_ind_17_bin     595212 non-null int64
ps_ind_18_bin     595212 non-null int64
ps_reg_01         595212 non-null float64
ps_reg_02         595212 non-null float64
ps_re

In [6]:
print(f'Dataset de treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
print(f'Dataset de teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')

Dataset de treino tem 595212 linhas por 59 colunas (35117508 celulas)
Dataset de teste tem 892816 linhas por 58 colunas (51783328 celulas)


## Observação
Com base na analise descritiva dos datasets, podemos perceber algumas coisas:

1 -  Os valores estão desnormalizados, variando tanto de tipo (numéricos discretos e continuos, categoricos e binários);

2 - Faltando dados em ambos os datasets (marcados como -1 nos datasets);

---

In [7]:
print(f'Antes - Treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
dfTreino.drop_duplicates()
print(f'Depois - Treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
print('---')
print(f'Antes - Teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')
dfTeste.drop_duplicates()
print(f'Depois - Teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')

Antes - Treino tem 595212 linhas por 59 colunas (35117508 celulas)
Depois - Treino tem 595212 linhas por 59 colunas (35117508 celulas)
---
Antes - Teste tem 892816 linhas por 58 colunas (51783328 celulas)
Depois - Teste tem 892816 linhas por 58 colunas (51783328 celulas)


Ao trabalhar com as colunas (atributos), é interessante ter uma organização de que tipo de dado determinado atributo é, e para quais propósitos determinado atributo pode ser usado. Nesse sentido, seguindo o trabalho de https://www.kaggle.com/bertcarremans/data-preparation-exploration, vamos criar metadados para esse conjunto.

In [8]:
def generateMetadata(dfInput):
    data = []
    for f in dfInput.columns:
        # definindo o uso (entre rótulo, id e atributos)
        if f == 'target':
            role = 'target' # rótulo
        elif f == 'id':
            role = 'id'
        else:
            role = 'input' # atributos

        # definindo o tipo do dado
        if 'bin' in f or f == 'target':
            level = 'binary'
        elif 'cat' in f or f == 'id':
            level = 'nominal'
        elif dfInput[f].dtype == float:
            level = 'interval'
        elif dfInput[f].dtype == int:
            level = 'ordinal'

        # mantem keep como verdadeiro pra tudo, exceto id
        keep = True
        if f == 'id':
            keep = False

        # cria o tipo de dado
        dtype = dfInput[f].dtype

        # cria dicionário de metadados
        f_dict = {
            'varname': f,
            'role': role,
            'level': level,
            'keep': keep,
            'dtype': dtype
        }
        data.append(f_dict)

    meta = pd.DataFrame(data, columns=['varname', 'role', 'level', 'keep', 'dtype'])
    meta.set_index('varname', inplace=True)
    
    return meta

Para visualizar o atributo e todos seus metadados, basta mostrar a variável meta:

In [9]:
meta_train = generateMetadata(dfTreino)
meta_test = generateMetadata(dfTeste)

In [10]:
display(meta_train)
display(meta_test)

,role,level,keep,dtype
varname,,,,
id,id,nominal,False,int64
target,target,binary,True,int64
ps_ind_01,input,binary,True,int64
ps_ind_02_cat,input,nominal,True,int64
ps_ind_03,input,nominal,True,int64
ps_ind_04_cat,input,nominal,True,int64
ps_ind_05_cat,input,nominal,True,int64
ps_ind_06_bin,input,binary,True,int64
ps_ind_07_bin,input,binary,True,int64


,role,level,keep,dtype
varname,,,,
id,id,nominal,False,int64
ps_ind_01,input,nominal,True,int64
ps_ind_02_cat,input,nominal,True,int64
ps_ind_03,input,nominal,True,int64
ps_ind_04_cat,input,nominal,True,int64
ps_ind_05_cat,input,nominal,True,int64
ps_ind_06_bin,input,binary,True,int64
ps_ind_07_bin,input,binary,True,int64
ps_ind_08_bin,input,binary,True,int64


Com essa estrutura de metadados, fica fácil consultar quais colunas quer se manter e que são nominais, por exemplo:

In [11]:
print('Metadados categoricos da base de treino')
print(meta_train[(meta_train.level == 'nominal') & (meta_train.keep)].index)
print('---')
print('Metadados categoricos da base de teste')
print(meta_test[(meta_test.level == 'nominal') & (meta_test.keep)].index)

Metadados categoricos da base de treino
Index(['ps_ind_02_cat', 'ps_ind_03', 'ps_ind_04_cat', 'ps_ind_05_cat',
       'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat',
       'ps_car_05_cat', 'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat',
       'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat', 'ps_car_11'],
      dtype='object', name='varname')
---
Metadados categoricos da base de teste
Index(['ps_ind_01', 'ps_ind_02_cat', 'ps_ind_03', 'ps_ind_04_cat',
       'ps_ind_05_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat',
       'ps_car_04_cat', 'ps_car_05_cat', 'ps_car_06_cat', 'ps_car_07_cat',
       'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat',
       'ps_car_11'],
      dtype='object', name='varname')


Da mesma forma, seria possível contar os atributos por tipo de uso e dado:

In [12]:
print('Tipos e quantidade de features do dataset de treino')
display(pd.DataFrame({'count' : meta_train.groupby(['role', 'level'])['role'].size()}).reset_index())

print('Tipos e quantidade de features do dataset de teste')
display(pd.DataFrame({'count' : meta_train.groupby(['role', 'level'])['role'].size()}).reset_index())

Tipos e quantidade de features do dataset de treino


,role,level,count
0,id,nominal,1
1,input,binary,20
2,input,interval,21
3,input,nominal,16
4,target,binary,1


Tipos e quantidade de features do dataset de teste


,role,level,count
0,id,nominal,1
1,input,binary,20
2,input,interval,21
3,input,nominal,16
4,target,binary,1


## Valores faltantes

Conforme já mencionado, os valores faltantes são indicados por -1, então é importante saber quais colunas têm valores faltantes e em qual proporção.

In [13]:
def getMissingAttributes(dfInput):
    atributos_missing = []
    return_missing = []

    for f in dfInput.columns:
        missings = dfInput[dfInput[f] == -1][f].count()
        if missings > 0:
            atributos_missing.append(f)
            missings_perc = missings/dfInput.shape[0]
            
            return_missing.append([f, missings, missings_perc])

            print('Atributo {} tem {} amostras ({:.2%}) com valores faltantes'.format(f, missings, missings_perc))
            

    print('No total, há {} atributos com valores faltantes'.format(len(atributos_missing)))
    
    return pd.DataFrame(return_missing).rename(index=str, columns={0: "column_name", 1: "column_nulls", 2: "column_percentage"})

In [14]:
missing_Train = getMissingAttributes(dfTreino)
display(missing_Train)

Atributo ps_ind_02_cat tem 216 amostras (0.04%) com valores faltantes
Atributo ps_ind_04_cat tem 83 amostras (0.01%) com valores faltantes
Atributo ps_ind_05_cat tem 5809 amostras (0.98%) com valores faltantes
Atributo ps_reg_03 tem 107772 amostras (18.11%) com valores faltantes
Atributo ps_car_01_cat tem 107 amostras (0.02%) com valores faltantes
Atributo ps_car_02_cat tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_03_cat tem 411231 amostras (69.09%) com valores faltantes
Atributo ps_car_05_cat tem 266551 amostras (44.78%) com valores faltantes
Atributo ps_car_07_cat tem 11489 amostras (1.93%) com valores faltantes
Atributo ps_car_09_cat tem 569 amostras (0.10%) com valores faltantes
Atributo ps_car_11 tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_12 tem 1 amostras (0.00%) com valores faltantes
Atributo ps_car_14 tem 42620 amostras (7.16%) com valores faltantes
No total, há 13 atributos com valores faltantes


,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,216,0.000363
1,ps_ind_04_cat,83,0.000139
2,ps_ind_05_cat,5809,0.009760
3,ps_reg_03,107772,0.181065
4,ps_car_01_cat,107,0.000180
5,ps_car_02_cat,5,0.000008
6,ps_car_03_cat,411231,0.690898
7,ps_car_05_cat,266551,0.447825
8,ps_car_07_cat,11489,0.019302
9,ps_car_09_cat,569,0.000956


In [15]:
missing_Test = getMissingAttributes(dfTeste)
display(missing_Test)

Atributo ps_ind_02_cat tem 307 amostras (0.03%) com valores faltantes
Atributo ps_ind_04_cat tem 145 amostras (0.02%) com valores faltantes
Atributo ps_ind_05_cat tem 8710 amostras (0.98%) com valores faltantes
Atributo ps_reg_03 tem 161684 amostras (18.11%) com valores faltantes
Atributo ps_car_01_cat tem 160 amostras (0.02%) com valores faltantes
Atributo ps_car_02_cat tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_03_cat tem 616911 amostras (69.10%) com valores faltantes
Atributo ps_car_05_cat tem 400359 amostras (44.84%) com valores faltantes
Atributo ps_car_07_cat tem 17331 amostras (1.94%) com valores faltantes
Atributo ps_car_09_cat tem 877 amostras (0.10%) com valores faltantes
Atributo ps_car_11 tem 1 amostras (0.00%) com valores faltantes
Atributo ps_car_14 tem 63805 amostras (7.15%) com valores faltantes
No total, há 12 atributos com valores faltantes


,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,307,0.000344
1,ps_ind_04_cat,145,0.000162
2,ps_ind_05_cat,8710,0.009756
3,ps_reg_03,161684,0.181094
4,ps_car_01_cat,160,0.000179
5,ps_car_02_cat,5,0.000006
6,ps_car_03_cat,616911,0.690972
7,ps_car_05_cat,400359,0.448423
8,ps_car_07_cat,17331,0.019412
9,ps_car_09_cat,877,0.000982


Duas estratégias podem ser optadas aqui: simplesmente remover o atributo ou tentar preenchê-lo de forma sintética. Preencher de forma sintética pode gerar uma falsa distribuição quando o número de atributos faltantes é muito alto. Quando este for o caso, é sempre seguro optar por remover o atributo inteiro. Também é importante lembrar que a estratégia de preenchimento deve ser coerente com o tipo de dado, por exemplo: **dados ordinais não devem ser preenchidos com média, nem dados contínuos com moda.**

In [16]:
# limiar de remoção - 42.5% de nulos
remove_threshold = 0.425

In [17]:
columns_to_remove = np.array(missing_Train.column_name[(missing_Train.column_percentage >= remove_threshold)])

In [18]:
# removendo as colunas que tem muitos valores faltantes
dfTreino = dfTreino.drop(columns_to_remove, axis=1)
dfTeste = dfTeste.drop(columns_to_remove, axis=1)

# atualiza os metadados para ter como referência
meta_train.loc[(columns_to_remove),'keep'] = False  
meta_test.loc[(columns_to_remove),'keep'] = False

# remove do frame de colunas com falta de dados as colunas que foram dropadas
missing_Train.drop(missing_Train[(np.isin(missing_Train.column_name, columns_to_remove))].index)

,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,216,0.000363
1,ps_ind_04_cat,83,0.000139
2,ps_ind_05_cat,5809,0.009760
3,ps_reg_03,107772,0.181065
4,ps_car_01_cat,107,0.000180
5,ps_car_02_cat,5,0.000008
8,ps_car_07_cat,11489,0.019302
9,ps_car_09_cat,569,0.000956
10,ps_car_11,5,0.000008
11,ps_car_12,1,0.000002


In [19]:
# Usa ou moda ou média para preencher os valores "vazios" que nosso dataset contem, baseado nos metadados do mesmo
def fillNullNumbers(dfInput, dfMetadata, dfMissing, missing_default):

    from sklearn.impute import SimpleImputer

    media_imp = SimpleImputer(missing_values=missing_default, strategy='mean')
    moda_imp = SimpleImputer(missing_values=missing_default, strategy='most_frequent')

    for index,row in dfMissing.iterrows():
        columnName = row['column_name']
        columnType = dfMetadata.level[(dfMetadata.index == columnName)][0]

        if (columnType == 'interval'):
            imputerToUse = media_imp
            imputerString = 'media_imp'
        elif (columnType == 'ordinal'):
            imputerToUse = moda_imp
            imputerString = 'moda_imp'
        else:
            imputerToUse = None
            imputerString = None

        if (imputerToUse != None):
            dfInput[columnName] = imputerToUse.fit_transform(dfInput[[columnName]]).ravel()
            print(f"Preenchendo coluna {columnName}, cujo tipo é {columnType}, usando o Imputer {imputerString}")

    return dfInput

In [20]:
dfTreino = fillNullNumbers(dfTreino, meta_train, missing_Train, -1)

ModuleNotFoundError: No module named 'sklearn.impute'

In [ ]:
from sklearn.impute import SimpleImputer

media_imp = SimpleImputer(missing_values=-1, strategy='mean')
moda_imp = SimpleImputer(missing_values=-1, strategy='most_frequent')

train['ps_reg_03'] = media_imp.fit_transform(train[['ps_reg_03']]).ravel()
train['ps_car_12'] = media_imp.fit_transform(train[['ps_car_12']]).ravel()
train['ps_car_14'] = media_imp.fit_transform(train[['ps_car_14']]).ravel()
train['ps_car_11'] = moda_imp.fit_transform(train[['ps_car_11']]).ravel()

test['ps_reg_03'] = media_imp.fit_transform(test[['ps_reg_03']]).ravel()
test['ps_car_12'] = media_imp.fit_transform(test[['ps_car_12']]).ravel()
test['ps_car_14'] = media_imp.fit_transform(test[['ps_car_14']]).ravel()
test['ps_car_11'] = moda_imp.fit_transform(test[['ps_car_11']]).ravel()

Os atributos categóricos podem ser mantidos porque o número de valores faltantes não é expressivo. Inclusive, a estratégia de preenchimento dos **atributos categóricos** é sempre mais complexa. Esses atributos **não se beneficiam de medidas estatísticas** como moda e média, portanto essas medidas não servem para preenchê-los de forma sintética.

---

## One-hot encoding (ou dummy variables)

Depois de ter tratado os dados faltantes, é importante que os dados ordinais tenham representação apropriada para o problema tratado. Se o dado não tem distância ou rankamento entre eles, cada valor de um atributo deve ser representado por um conjunto de atributos de mesma distância. *(Verificar slides desse encontro para que isso fique mais claro)*

Os dados que precisam ser separados em mais dimensões já foram identificados como nominais no pré-processamento. É importante verificar se esses dados têm grande variedade de valores ou não, e aplicar essa separação apenas se for viável. Por exemplo, se um determinado atributo tem 300 valores, isso geraria 300 colunas novas. Isso só se justificaria se fosse uma base realmente grande e se houvesse uma correlação muito alta entre essa variedade de valores e a classe.

In [ ]:
v = meta[(meta.level == 'nominal') & (meta.keep)].index

for f in v:
    dist_values = train[f].value_counts().shape[0]
    print('Atributo {} tem {} valores distintos'.format(f, dist_values))

Vamos optar por manter todos atributos e, portanto, gerar o conjunto de atributos que os mantêm à mesma distância:

In [ ]:
v = meta[(meta.level == 'nominal') & (meta.keep)].index
print('Antes do one-hot encoding tinha-se {} atributos'.format(train.shape[1]))
train = pd.get_dummies(train, columns=v, drop_first=True)
print('Depois do one-hot encoding tem-se {} atributos'.format(train.shape[1]))

test = pd.get_dummies(test, columns=v, drop_first=True)
missing_cols = set( train.columns ) - set( test.columns )
for c in missing_cols:
    test[c] = 0
    
train, test = train.align(test, axis=1)

## Depois de todo pré-processamento...

É hora de verificar se tanto treino como teste têm o mesmo tamanho/formato, e aplicar um modelo de classificação já que esse é um problema desse tipo. Vale lembrar que o tamanho do treino e teste pode variar quando você estiver participando de outras competições ou explorando outros conjuntos de dados.

Isso porque na maioria das competições não se tem o *target* do test. Estima-se uma resposta e submete ao Kaggle, por exemplo, para que ele verifique qual foi o resultado final. Então esse tamanho pode variar em 1 entre treino e teste. No nosso caso, como todos os dados vêm de uma mesma fonte para experimentos, é esperado que tenham a mesma quantidade de atributos ou colunas.

In [ ]:
print(train.shape)
print(test.shape)

In [ ]:
X_train = train.drop(['id', 'target'], axis=1)
y_train = train['target']

X_test  = test.drop(['id', 'target'], axis=1)
y_test  = test['target']

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)
model.score(X_test, y_test)

## Exercícios

1. A partir desse ponto, verifique a distribuição das classes e analise qual métrica seria mais interessante para estimar o resultado de forma justa.
2. Utilize um método de classificação da sua escolha e aplique hiperparametrização.
3. Verifique qual o resultado que se tem utilizando validação cruzada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager
from sklearn import svm

xx, yy = np.meshgrid(np.linspace(-5, 5, 500), np.linspace(-5, 5, 500))
# Generate train data
X = 0.3 * np.random.randn(100, 2)
X_train = np.r_[X + 2, X - 2]
# Generate some regular novel observations
X = 0.3 * np.random.randn(20, 2)
X_test = np.r_[X + 2, X - 2]
# Generate some abnormal novel observations
X_outliers = np.random.uniform(low=-4, high=4, size=(20, 2))

# fit the model
clf = svm.OneClassSVM(nu=0.1, kernel="rbf", gamma=0.1)
clf.fit(X_train)
y_pred_train = clf.predict(X_train)
y_pred_test = clf.predict(X_val)
y_pred_outliers = clf.predict(X_outliers)
n_error_train = y_pred_train[y_pred_train == -1].size
n_error_test = y_pred_test[y_pred_test == -1].size
n_error_outliers = y_pred_outliers[y_pred_outliers == 1].size

# plot the line, the points, and the nearest vectors to the plane
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.title("Novelty Detection")
plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
a = plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='darkred')
plt.contourf(xx, yy, Z, levels=[0, Z.max()], colors='palevioletred')

s = 40
b1 = plt.scatter(X_train[:, 0], X_train[:, 1], c='white', s=s, edgecolors='k')
b2 = plt.scatter(X_test[:, 0], X_test[:, 1], c='blueviolet', s=s,
                 edgecolors='k')
c = plt.scatter(X_outliers[:, 0], X_outliers[:, 1], c='gold', s=s,
                edgecolors='k')
plt.axis('tight')
plt.xlim((-5, 5))
plt.ylim((-5, 5))
plt.legend([a.collections[0], b1, b2, c],
           ["learned frontier", "training observations",
            "new regular observations", "new abnormal observations"],
           loc="upper left",
           prop=matplotlib.font_manager.FontProperties(size=11))
plt.xlabel(
    "error train: %d/200 ; errors novel regular: %d/40 ; "
    "errors novel abnormal: %d/40"
    % (n_error_train, n_error_test, n_error_outliers))
plt.show()